# Fink/LSST — Reload Dipole Analysis per DDF (fully offline)

This notebook is the **fully-offline reload** variant of `01c_fink_dipoles_per_ddf.ipynb`.

All alert data are read from the parquet files already written by `01c` and stored in
`data_DIPOLES_01c/`.  **No API call is made anywhere in this notebook.**

## Files read

| File | Content |
|------|---------|
| `data_DIPOLES_01c/{field}_alerts.parquet` | All alerts for one DDF (one row per alert) |
| `data_DIPOLES_01c/all_ddfs_dipoles_only.parquet` | Concatenated dipole-only alerts (all DDFs) |

## Figures reproduced (same as `01c`)

1. Summary statistics table per DDF  
2. Sky maps with dipole orientation arrows per DDF  
3. Rose diagrams (polar + linear) of dipole position angle, stacked by band  
4. Dipole fraction vs time (per DDF, independent x-axis)  
5. Dipole fraction vs time (all DDFs stitched, shared x-axis)  
6. Dipole fraction per visit and per band (per DDF)  
7. Dipole morphology distributions (length, angle, chi2, fluxDiff — all DDFs combined)  
8. Dipole angle polar histogram (all DDFs, stacked by band)  
9. Dipole fraction per DDF × band heatmap  

Output figures go to `figs_DIPOLES_01d/`.


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-28
- last update : 2026-05-28

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch
from astropy.time import Time

warnings.filterwarnings("ignore")
print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Input: parquet files written by 01c ──────────────────────────────────────
DIR_DATA_IN = "data_DIPOLES_01c"

# ── Output figures (separate directory to avoid overwriting 01c figures) ──────
NB_TAG = "DIPOLES_01d"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Input data : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figures    : {os.path.abspath(DIR_FIGS)}")

# ── LSST Deep Drilling Fields (same order as 01c) ─────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Plotting style ────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER_LOCAL = list("ugrizy")
MJD_BIN_DAYS = 1  # bin width for dipole-fraction-vs-time (days)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Load parquet files from `data_DIPOLES_01c`

One parquet file per DDF; no API call.

In [ ]:
ddf_alerts: dict[str, pd.DataFrame] = {}

for field_name in DEEP_FIELDS:
    pq = os.path.join(DIR_DATA_IN, f"{field_name}_alerts.parquet")
    if not os.path.exists(pq):
        print(f"[{field_name:12s}] parquet not found — skipping.")
        ddf_alerts[field_name] = pd.DataFrame()
        continue
    df = pd.read_parquet(pq)

    # Ensure proper types
    for bool_col in ("r:isDipole", "r:isNegative", "r:dipoleFitAttempted"):
        if bool_col in df.columns:
            df[bool_col] = (
                df[bool_col]
                .map(
                    lambda v: (
                        True
                        if str(v).strip().lower() in ("true", "1", "yes")
                        else False
                        if str(v).strip().lower() in ("false", "0", "no")
                        else pd.NA
                    )
                )
                .astype("boolean")
            )
    for num_col in (
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:nDiaSources",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:scienceFlux",
        "r:scienceFluxErr",
        "r:templateFlux",
        "r:templateFluxErr",
        "r:apFlux",
        "r:apFluxErr",
        "r:dipoleFluxDiff",
        "r:dipoleFluxDiffErr",
        "r:dipoleMeanFlux",
        "r:dipoleMeanFluxErr",
        "r:dipoleLength",
        "r:dipoleAngle",
        "r:dipoleChi2",
        "r:dipoleNdata",
    ):
        if num_col in df.columns:
            df[num_col] = pd.to_numeric(df[num_col], errors="coerce")

    ddf_alerts[field_name] = df
    n_dip = int(df["r:isDipole"].fillna(False).sum()) if "r:isDipole" in df.columns else 0
    frac = n_dip / len(df) * 100 if len(df) > 0 else 0.0
    print(f"[{field_name:12s}] {len(df):7,} alerts  |  {n_dip:6,} dipoles  ({frac:.2f}%)")

print("\nLoad complete.")

In [ ]:
# ── Load concatenated dipole-only catalogue ───────────────────────────────────
pq_all = os.path.join(DIR_DATA_IN, "all_ddfs_dipoles_only.parquet")
if os.path.exists(pq_all):
    df_all_dipoles = pd.read_parquet(pq_all)
    for num_col in (
        "r:dipoleAngle",
        "r:dipoleLength",
        "r:dipoleChi2",
        "r:dipoleFluxDiff",
        "r:midpointMjdTai",
    ):
        if num_col in df_all_dipoles.columns:
            df_all_dipoles[num_col] = pd.to_numeric(df_all_dipoles[num_col], errors="coerce")
    print(f"all_ddfs_dipoles_only: {len(df_all_dipoles):,} rows")
    if "field" in df_all_dipoles.columns:
        print(df_all_dipoles["field"].value_counts().to_string())
else:
    # Rebuild from per-DDF parquets if not present
    frames = []
    for field_name, df in ddf_alerts.items():
        if df.empty or "r:isDipole" not in df.columns:
            continue
        df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
        df_dip["field"] = field_name
        frames.append(df_dip)
    df_all_dipoles = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    print(f"[rebuilt] all_ddfs_dipoles_only: {len(df_all_dipoles):,} rows")

## 3. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """Convert an array of MJD (TAI) values to ISO date strings 'YYYY-MM-DD'."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 7) -> None:
    """Add a secondary x-axis on top of *ax* showing calendar dates (YYYY-MM-DD)."""
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return
    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return
    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=7, labelpad=6)


def compute_dipole_fraction_vs_time(
    df: pd.DataFrame,
    bin_days: float = MJD_BIN_DAYS,
    mjd_col: str = "r:midpointMjdTai",
    flag_col: str = "r:isDipole",
) -> pd.DataFrame:
    """Compute dipole fraction in time bins."""
    if df.empty or mjd_col not in df.columns or flag_col not in df.columns:
        return pd.DataFrame()
    df = df.copy()
    df[flag_col] = df[flag_col].fillna(False).astype(bool)
    df[mjd_col] = pd.to_numeric(df[mjd_col], errors="coerce")
    df = df.dropna(subset=[mjd_col])
    if df.empty:
        return pd.DataFrame()
    mjd_min = df[mjd_col].min()
    mjd_max = df[mjd_col].max()
    bins = np.arange(mjd_min, mjd_max + bin_days, bin_days)
    df["mjd_bin"] = pd.cut(df[mjd_col], bins=bins, labels=False)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0
    rows = []
    for i, center in enumerate(bin_centers):
        sub = df[df["mjd_bin"] == i]
        n_tot = len(sub)
        n_dip = int(sub[flag_col].sum())
        frac = n_dip / n_tot if n_tot > 0 else np.nan
        frac_e = np.sqrt(n_dip) / n_tot if (n_tot > 0 and n_dip > 0) else np.nan
        rows.append(
            {
                "mjd_bin_center": center,
                "n_total": n_tot,
                "n_dipoles": n_dip,
                "dipole_fraction": frac,
                "dipole_fraction_err": frac_e,
            }
        )
    return pd.DataFrame(rows)


def compute_dipole_fraction_per_visit(
    df: pd.DataFrame,
    visit_col: str = "r:visit",
    flag_col: str = "r:isDipole",
    mjd_col: str = "r:midpointMjdTai",
    band_col: str = "r:band",
) -> pd.DataFrame:
    """Compute dipole fraction per (visit, band) pair."""
    if df.empty or visit_col not in df.columns or flag_col not in df.columns:
        return pd.DataFrame()
    df = df.copy()
    df[flag_col] = df[flag_col].fillna(False).astype(bool)
    group_cols = [visit_col] + ([band_col] if band_col in df.columns else [])
    agg_dict = {"n_total": (flag_col, "count"), "n_dipoles": (flag_col, "sum")}
    if mjd_col in df.columns:
        agg_dict["mjd_mean"] = (mjd_col, "mean")
    agg = df.groupby(group_cols).agg(**agg_dict).reset_index()
    agg["dipole_fraction"] = agg["n_dipoles"] / agg["n_total"]
    return agg.sort_values(visit_col).reset_index(drop=True)


print("Utility functions defined.")

## 4. Summary statistics per DDF

In [ ]:
rows = []
for field_name, df in ddf_alerts.items():
    if df.empty:
        rows.append(
            {
                "field": field_name,
                "n_alerts": 0,
                "n_dipoles": 0,
                "dipole_fraction": np.nan,
                "n_fit_attempted": 0,
            }
        )
        continue
    n_tot = len(df)
    n_dip = int(df["r:isDipole"].fillna(False).astype(bool).sum()) if "r:isDipole" in df.columns else 0
    n_fit = (
        int(df["r:dipoleFitAttempted"].fillna(False).astype(bool).sum())
        if "r:dipoleFitAttempted" in df.columns
        else 0
    )
    rows.append(
        {
            "field": field_name,
            "n_alerts": n_tot,
            "n_dipoles": n_dip,
            "dipole_fraction": n_dip / n_tot if n_tot else np.nan,
            "n_fit_attempted": n_fit,
        }
    )

df_summary = pd.DataFrame(rows)
print("Summary statistics per DDF:")
display(df_summary)

## 5. Sky maps with dipole orientation arrows per DDF

In [ ]:
def plot_skymap_dipoles(ddf_alerts: dict, arrow_scale: float = 0.02) -> None:
    """
    One panel per DDF: scatter of all alerts (grey) + dipole alerts (coloured by band)
    with a FancyArrow indicating the dipole position angle.
    """
    n_fields = len(DEEP_FIELDS)
    ncols = min(3, n_fields)
    nrows = int(np.ceil(n_fields / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)

    for idx, (field_name, (ra_c, dec_c)) in enumerate(DEEP_FIELDS.items()):
        ax = axes[idx // ncols][idx % ncols]
        df = ddf_alerts.get(field_name, pd.DataFrame())

        if df.empty or "r:ra" not in df.columns:
            ax.set_title(f"{field_name} — no data")
            continue

        ra = pd.to_numeric(df["r:ra"], errors="coerce")
        dec = pd.to_numeric(df["r:dec"], errors="coerce")

        # All alerts in light grey
        ax.scatter(ra, dec, s=1, color="lightgrey", alpha=0.4, rasterized=True, label="all alerts")

        # Dipole alerts per band
        if "r:isDipole" in df.columns:
            df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
            if not df_dip.empty:
                bands = df_dip["r:band"].dropna().unique() if "r:band" in df_dip.columns else []
                for band in BAND_ORDER_LOCAL:
                    if band not in bands:
                        continue
                    sub = df_dip[df_dip["r:band"] == band]
                    ra_d = pd.to_numeric(sub["r:ra"], errors="coerce").values
                    dec_d = pd.to_numeric(sub["r:dec"], errors="coerce").values
                    ax.scatter(
                        ra_d,
                        dec_d,
                        s=8,
                        color=BAND_COLORS[band],
                        alpha=0.7,
                        label=f"dipole {band} (n={len(sub):,})",
                        zorder=3,
                    )

                # Arrows for dipole orientation (subsample if many)
                if "r:dipoleAngle" in df_dip.columns:
                    sub_arrow = df_dip.dropna(subset=["r:ra", "r:dec", "r:dipoleAngle"]).copy()
                    if len(sub_arrow) > 300:
                        sub_arrow = sub_arrow.sample(300, random_state=42)
                    for _, row in sub_arrow.iterrows():
                        pa_deg = float(row["r:dipoleAngle"])
                        pa_rad = np.radians(pa_deg)
                        dx = arrow_scale * np.sin(pa_rad)  # RA increases East
                        dy = arrow_scale * np.cos(pa_rad)  # Dec increases North
                        ax.annotate(
                            "",
                            xy=(float(row["r:ra"]) + dx, float(row["r:dec"]) + dy),
                            xytext=(float(row["r:ra"]) - dx, float(row["r:dec"]) - dy),
                            arrowprops=dict(arrowstyle="->", color="k", lw=0.5),
                        )

        ax.scatter(ra_c, dec_c, marker="+", s=120, color="red", lw=2, zorder=5, label="field centre")
        ax.set_xlabel("RA (deg)")
        ax.set_ylabel("Dec (deg)")
        ax.set_title(f"{field_name} — sky map", fontsize=9)
        ax.invert_xaxis()  # RA increases to the left
        ax.legend(loc="best", fontsize=6, ncol=2, markerscale=2)

    for idx in range(n_fields, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle("Dipole sky maps per DDF  (arrows = dipole position angle)", fontsize=11, y=1.005)
    plt.tight_layout()
    savefig("dipole_skymap_per_ddf")
    plt.show()


plot_skymap_dipoles(ddf_alerts)

## 6. Rose diagrams of dipole position angle per DDF

Left panel: polar rose diagram.  Right panel: linear histogram stacked by band.

In [ ]:
def plot_dipole_direction_per_ddf(ddf_alerts: dict) -> None:
    """Rose (polar) + linear histogram of dipole PA per DDF, stacked by band."""
    angle_col = "r:dipoleAngle"
    band_col = "r:band"
    n_bins_rose = 36
    bin_edges_rad = np.linspace(0, 2 * np.pi, n_bins_rose + 1)
    bin_centers = (bin_edges_rad[:-1] + bin_edges_rad[1:]) / 2.0
    width = 2 * np.pi / n_bins_rose
    lin_edges = np.linspace(0, 360, n_bins_rose + 1)

    n_fields = sum(1 for df in ddf_alerts.values() if not df.empty)
    if n_fields == 0:
        print("No data — skipping rose diagrams.")
        return

    fig = plt.figure(figsize=(12, 4.5 * n_fields))

    row = -1
    for field_name, df in ddf_alerts.items():
        if df.empty or "r:isDipole" not in df.columns:
            continue
        df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
        row += 1

        angles_deg = (
            pd.to_numeric(df_dip.get(angle_col, pd.Series(dtype=float)), errors="coerce").dropna().values
            % 360.0
        )
        n_total_dip = len(angles_deg)

        # ── Left: polar rose ──────────────────────────────────────────────────
        ax_pol = fig.add_subplot(n_fields, 2, 2 * row + 1, projection="polar")

        if band_col in df_dip.columns and n_total_dip > 0:
            bands_present = [b for b in BAND_ORDER_LOCAL if b in df_dip[band_col].dropna().unique()]
            bottom_pol = np.zeros(n_bins_rose)
            for band in bands_present:
                sub_rad = np.radians(
                    pd.to_numeric(df_dip.loc[df_dip[band_col] == band, angle_col], errors="coerce")
                    .dropna()
                    .values
                    % 360.0
                )
                if len(sub_rad) == 0:
                    continue
                cnts, _ = np.histogram(sub_rad, bins=bin_edges_rad)
                ax_pol.bar(
                    bin_centers,
                    cnts,
                    width=width * 0.9,
                    bottom=bottom_pol,
                    color=BAND_COLORS.get(band, "grey"),
                    edgecolor="white",
                    linewidth=0.3,
                    alpha=0.85,
                    label=f"{band} (n={len(sub_rad):,})",
                )
                bottom_pol += cnts
            uniform_level = n_total_dip / n_bins_rose
            ax_pol.plot(
                np.linspace(0, 2 * np.pi, 300),
                np.full(300, uniform_level),
                "--",
                color="crimson",
                lw=1.0,
                alpha=0.8,
                label="uniform",
            )
            ax_pol.legend(loc="lower right", fontsize=6, bbox_to_anchor=(1.28, -0.05))
        elif n_total_dip > 0:
            cnts, _ = np.histogram(np.radians(angles_deg), bins=bin_edges_rad)
            ax_pol.bar(
                bin_centers,
                cnts,
                width=width * 0.9,
                color="steelblue",
                edgecolor="white",
                linewidth=0.3,
                alpha=0.8,
            )

        ax_pol.set_theta_zero_location("N")
        ax_pol.set_theta_direction(-1)
        ax_pol.tick_params(labelsize=7)
        ax_pol.set_title(f"{field_name} — rose diagram (n={n_total_dip:,})", va="bottom", pad=18, fontsize=9)
        for pa_deg, label in [(0, "N"), (90, "E"), (180, "S"), (270, "W")]:
            ax_pol.text(
                np.radians(pa_deg),
                ax_pol.get_rmax() * 1.2,
                label,
                ha="center",
                va="center",
                fontsize=7,
                fontweight="bold",
            )

        # ── Right: linear histogram by band ───────────────────────────────────
        ax_lin = fig.add_subplot(n_fields, 2, 2 * row + 2)
        if n_total_dip == 0:
            ax_lin.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax_lin.transAxes)
        elif band_col in df_dip.columns:
            bottom_lin = np.zeros(n_bins_rose)
            for band in [b for b in BAND_ORDER_LOCAL if b in df_dip[band_col].dropna().unique()]:
                sub = (
                    pd.to_numeric(df_dip.loc[df_dip[band_col] == band, angle_col], errors="coerce")
                    .dropna()
                    .values
                    % 360.0
                )
                if len(sub) == 0:
                    continue
                cnts, _ = np.histogram(sub, bins=lin_edges)
                ax_lin.bar(
                    lin_edges[:-1],
                    cnts,
                    width=360.0 / n_bins_rose * 0.9,
                    bottom=bottom_lin,
                    color=BAND_COLORS.get(band, "grey"),
                    edgecolor="white",
                    linewidth=0.3,
                    label=f"{band} (n={len(sub):,})",
                    alpha=0.85,
                )
                bottom_lin += cnts
        else:
            cnts, _ = np.histogram(angles_deg, bins=lin_edges)
            ax_lin.bar(
                lin_edges[:-1],
                cnts,
                width=360.0 / n_bins_rose * 0.9,
                color="steelblue",
                edgecolor="white",
                linewidth=0.3,
            )

        ax_lin.set_xlabel("Dipole PA (deg, N through E)")
        ax_lin.set_ylabel("N dipole alerts")
        ax_lin.set_xlim(0, 360)
        ax_lin.set_xticks(np.arange(0, 361, 45))
        ax_lin.set_xticklabels(
            [
                "N\n0deg",
                "45deg",
                "E\n90deg",
                "135deg",
                "S\n180deg",
                "225deg",
                "W\n270deg",
                "315deg",
                "N\n360deg",
            ],
            fontsize=7,
        )
        ax_lin.set_title(f"{field_name} — dipoleAngle by band", fontsize=9)
        ax_lin.legend(loc="upper right", fontsize=7, ncol=2)

    fig.suptitle("Dipole orientation per DDF", fontsize=11, y=1.005)
    plt.tight_layout()
    savefig("dipole_direction_per_ddf")
    plt.show()


plot_dipole_direction_per_ddf(ddf_alerts)

## 7. Dipole fraction vs time (per DDF, independent x-axis)

In [ ]:
fig, axes = plt.subplots(nrows=len(DEEP_FIELDS), ncols=1, figsize=(11, 2.5 * len(DEEP_FIELDS)), sharex=False)
if len(DEEP_FIELDS) == 1:
    axes = [axes]

for ax, (field_name, df) in zip(axes, ddf_alerts.items()):
    df_time = compute_dipole_fraction_vs_time(df, bin_days=MJD_BIN_DAYS)
    if df_time.empty:
        ax.set_title(f"{field_name} — no data")
        continue
    mjd = df_time["mjd_bin_center"].values
    frac = df_time["dipole_fraction"].values * 100.0
    err = df_time["dipole_fraction_err"].fillna(0).values * 100.0
    ax.errorbar(
        mjd,
        frac,
        yerr=err,
        fmt="o-",
        ms=4,
        lw=1.2,
        capsize=3,
        color="steelblue",
        label=f"bin={MJD_BIN_DAYS}d",
    )
    ax2 = ax.twinx()
    ax2.bar(
        mjd, df_time["n_total"].values, width=MJD_BIN_DAYS * 0.8, color="grey", alpha=0.4, label="N alerts"
    )
    ax2.set_ylabel("N alerts", fontsize=7, color="grey")
    ax2.tick_params(axis="y", labelcolor="grey", labelsize=8)
    ax2.set_yscale("log")
    ax.set_ylabel("Dipole fraction (%)")
    ax.set_xlabel("MJD (TAI)")
    ax.set_title(f"{field_name} — dipole fraction vs time (bin={MJD_BIN_DAYS} days)")
    ax.legend(loc="upper right", fontsize=8)
    add_date_axis_on_top(ax, mjd, n_ticks=10)

plt.tight_layout()
savefig(f"dipole_fraction_vs_time_bin{MJD_BIN_DAYS}d")
plt.show()

## 8. Dipole fraction vs time (all DDFs stitched, shared x-axis)

In [ ]:
fig, axes = plt.subplots(nrows=len(DEEP_FIELDS), ncols=1, figsize=(11, 2.0 * len(DEEP_FIELDS)), sharex=True)
if len(DEEP_FIELDS) == 1:
    axes = [axes]

for ddf_count, (ax, (field_name, df)) in enumerate(zip(axes, ddf_alerts.items()), start=1):
    df_time = compute_dipole_fraction_vs_time(df, bin_days=MJD_BIN_DAYS)
    if df_time.empty:
        ax.set_title(f"{field_name} — no data")
        continue
    mjd = df_time["mjd_bin_center"].values
    frac = df_time["dipole_fraction"].values * 100.0
    err = df_time["dipole_fraction_err"].fillna(0).values * 100.0
    ax.errorbar(
        mjd,
        frac,
        yerr=err,
        fmt="o-",
        ms=4,
        lw=1.2,
        capsize=3,
        color="steelblue",
        label=f"bin={MJD_BIN_DAYS}d",
    )
    ax2 = ax.twinx()
    ax2.bar(mjd, df_time["n_total"].values, width=MJD_BIN_DAYS * 0.8, color="grey", alpha=0.4)
    ax2.set_ylabel("N alerts", fontsize=7, color="grey")
    ax2.tick_params(axis="y", labelcolor="grey", labelsize=8)
    ax2.set_yscale("log")
    ax.set_ylabel("Dipole frac (%)")
    ax.text(
        0.05,
        0.92,
        f" DDF : {field_name}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        bbox=dict(boxstyle="round", fc="white", alpha=0.5),
    )
    if ddf_count == 1:
        ax.set_title(f"Dipole fraction vs time (bin={MJD_BIN_DAYS} days)")
        add_date_axis_on_top(ax, mjd, n_ticks=10)
    if ddf_count == len(DEEP_FIELDS):
        ax.set_xlabel("MJD (TAI)")

plt.tight_layout()
savefig(f"dipole_fraction_vs_time_bin{MJD_BIN_DAYS}d_stitched")
plt.show()

## 9. Dipole fraction per visit and per band (per DDF)

In [ ]:
for field_name, df in ddf_alerts.items():
    df_visit = compute_dipole_fraction_per_visit(df)
    if df_visit.empty:
        print(f"[{field_name}] No per-visit data — skipping.")
        continue

    fig, ax = plt.subplots(figsize=(12, 4))
    if "r:band" in df_visit.columns:
        for band, grp in df_visit.groupby("r:band"):
            ax.plot(
                grp["r:visit"].astype(str),
                grp["dipole_fraction"] * 100,
                "o-",
                ms=3,
                lw=1,
                color=BAND_COLORS.get(band, "grey"),
                label=f"band={band}",
            )
    else:
        ax.plot(
            df_visit["r:visit"].astype(str),
            df_visit["dipole_fraction"] * 100,
            "o-",
            ms=3,
            lw=1,
            color="steelblue",
        )

    ax.xaxis.set_major_locator(plt.MaxNLocator(20))
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.set_xlabel("Visit ID")
    ax.set_ylabel("Dipole fraction (%)")
    ax.set_title(f"{field_name} — dipole fraction per visit")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    savefig(f"dipole_fraction_per_visit_{field_name.replace('-', '_')}")
    plt.show()

## 10. Dipole morphology distributions (all DDFs combined)

Distributions of `dipoleLength`, `dipoleAngle`, `dipoleChi2`, and `dipoleFluxDiff`.

In [ ]:
if df_all_dipoles.empty:
    print("No dipole data available — skipping morphology plots.")
else:
    morph_cols = {
        "r:dipoleLength": ("Dipole length (arcsec)", 0, 20, 50),
        "r:dipoleAngle": ("Dipole angle (deg)", 0, 360, 36),
        "r:dipoleChi2": ("Dipole chi2", 0, 50, 50),
        "r:dipoleFluxDiff": ("Dipole flux diff (nJy)", None, None, 60),
    }
    fig, axes = plt.subplots(1, len(morph_cols), figsize=(14, 4))
    for ax, (col, (xlabel, xmin, xmax, nbins)) in zip(axes, morph_cols.items()):
        if col not in df_all_dipoles.columns:
            ax.set_title(f"{col}\n(not found)")
            continue
        vals = pd.to_numeric(df_all_dipoles[col], errors="coerce").dropna().values
        if xmin is not None and xmax is not None:
            vals = vals[(vals >= xmin) & (vals <= xmax)]
        ax.hist(vals, bins=nbins, color="steelblue", edgecolor="white", linewidth=0.3)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("N alerts")
        ax.set_title(f"{col.split(':')[1]}\n(n={len(vals):,})")
    plt.suptitle("Dipole morphology — all DDFs combined", y=1.02, fontsize=11)
    plt.tight_layout()
    savefig("dipole_morphology_all_ddfs")
    plt.show()

## 11. Dipole angle polar histogram (all DDFs, stacked by band)

In [ ]:
if df_all_dipoles.empty or "r:dipoleAngle" not in df_all_dipoles.columns:
    print("Skipping polar histogram — no dipoleAngle data.")
else:
    n_bins = 36
    bin_edges_rad = np.linspace(0, 2 * np.pi, n_bins + 1)
    bin_centers = (bin_edges_rad[:-1] + bin_edges_rad[1:]) / 2.0
    width = 2 * np.pi / n_bins

    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection="polar")
    n_total_dip = 0

    if "r:band" in df_all_dipoles.columns:
        bands_present = [b for b in BAND_ORDER_LOCAL if b in df_all_dipoles["r:band"].dropna().unique()]
        bottom_pol = np.zeros(n_bins)
        for band in bands_present:
            sub_rad = np.radians(
                pd.to_numeric(
                    df_all_dipoles.loc[df_all_dipoles["r:band"] == band, "r:dipoleAngle"], errors="coerce"
                )
                .dropna()
                .values
                % 360.0
            )
            if len(sub_rad) == 0:
                continue
            n_total_dip += len(sub_rad)
            cnts, _ = np.histogram(sub_rad, bins=bin_edges_rad)
            ax.bar(
                bin_centers,
                cnts,
                width=width * 0.9,
                bottom=bottom_pol,
                color=BAND_COLORS.get(band, "grey"),
                edgecolor="white",
                linewidth=0.4,
                alpha=0.85,
                label=f"{band} (n={len(sub_rad):,})",
            )
            bottom_pol += cnts
        ax.legend(loc="lower right", fontsize=7, bbox_to_anchor=(1.30, -0.05))
    else:
        angles_rad = np.radians(
            pd.to_numeric(df_all_dipoles["r:dipoleAngle"], errors="coerce").dropna().values % 360.0
        )
        n_total_dip = len(angles_rad)
        cnts, _ = np.histogram(angles_rad, bins=bin_edges_rad)
        ax.bar(
            bin_centers,
            cnts,
            width=width * 0.9,
            color="steelblue",
            edgecolor="white",
            linewidth=0.4,
            alpha=0.8,
        )

    if n_total_dip > 0:
        uniform_val = n_total_dip / n_bins
        ax.plot(
            np.linspace(0, 2 * np.pi, 300),
            np.full(300, uniform_val),
            "--",
            color="crimson",
            lw=1.2,
            alpha=0.85,
            label="uniform",
        )

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(f"Dipole angle distribution\n(all DDFs, n={n_total_dip:,})", va="bottom", pad=20)
    plt.tight_layout()
    savefig("dipole_angle_polar_histogram")
    plt.show()

## 12. Dipole fraction per DDF × band heatmap

In [ ]:
band_rows = []
for field_name, df in ddf_alerts.items():
    if df.empty or "r:isDipole" not in df.columns or "r:band" not in df.columns:
        continue
    df = df.copy()
    df["r:isDipole"] = df["r:isDipole"].fillna(False).astype(bool)
    for band, grp in df.groupby("r:band"):
        n_tot = len(grp)
        n_dip = int(grp["r:isDipole"].sum())
        band_rows.append(
            {
                "field": field_name,
                "band": band,
                "n_total": n_tot,
                "n_dipoles": n_dip,
                "dipole_fraction": n_dip / n_tot if n_tot > 0 else np.nan,
            }
        )

df_band = pd.DataFrame(band_rows)
if not df_band.empty:
    pivot = df_band.pivot_table(index="field", columns="band", values="dipole_fraction").reindex(
        columns=list("ugrizy")
    )
    print("Dipole fraction per field × band:")
    print(pivot.to_string(float_format="{:.4f}".format))

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(pivot.values * 100, aspect="auto", cmap="YlOrRd", vmin=0)
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns.tolist())
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index.tolist())
    ax.set_xlabel("Band")
    ax.set_ylabel("DDF")
    ax.set_title("Dipole fraction (%) per DDF × band")
    plt.colorbar(im, ax=ax, label="Dipole fraction (%)")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val * 100:.1f}", ha="center", va="center", fontsize=8, color="black")
    plt.tight_layout()
    savefig("dipole_fraction_field_vs_band_heatmap")
    plt.show()
else:
    print("No band data available — skipping heatmap.")

## 13. Summary

| Step | Source |
|------|--------|
| Per-DDF alert catalogues | `data_DIPOLES_01c/{field}_alerts.parquet` |
| Dipole-only catalogue | `data_DIPOLES_01c/all_ddfs_dipoles_only.parquet` |
| **No API call made** | — |

All figures saved to `figs_DIPOLES_01d/`.
